In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ─── Estados del autómata celular ────────────────────────────────────────────
# Cada celda del bosque puede estar en uno de 4 estados:
EMPTY    = 0  # tierra sin vegetación / cortafuego
TREE     = 1  # árbol sano
BURNING  = 2  # árbol en llamas
BURNED   = 3  # árbol quemado (ya no propaga fuego)

# Paleta de colores para visualización
CMAP = mcolors.ListedColormap(['#c2b280', '#2d6a1f', '#ff4500', '#1a1a1a'])
NORM = mcolors.BoundaryNorm([0, 1, 2, 3, 4], CMAP.N)


def forest_step(grid, p_spread=0.6, p_die=0.9, wind=(0, 0)):
    """
    Avanza el autómata celular del incendio un paso.

    Reglas:
      - BURNING → BURNED con probabilidad p_die (el árbol se consume)
      - TREE vecino de BURNING → BURNING con probabilidad p_spread
        (el viento sesga la probabilidad hacia la dirección wind)
      - EMPTY y BURNED no cambian

    Parámetros
    ----------
    p_spread : probabilidad base de que un árbol sano se encienda
    p_die    : probabilidad de que un árbol en llamas se consuma en este paso
    wind     : tupla (dy, dx) con dirección del viento; valores típicos -1/0/1
    """
    new_grid = grid.copy()
    rows, cols = grid.shape

    for r in range(rows):
        for c in range(cols):

            if grid[r, c] == BURNING:
                # El árbol en llamas se consume con probabilidad p_die
                if np.random.rand() < p_die:
                    new_grid[r, c] = BURNED

            elif grid[r, c] == TREE:
                # Revisa los 8 vecinos en busca de fuego
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        if dr == 0 and dc == 0:
                            continue
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] == BURNING:
                            # El viento aumenta la probabilidad si sopla
                            # hacia este vecino (producto punto positivo)
                            wind_bonus = 0.15 * (dr * wind[0] + dc * wind[1])
                            prob = np.clip(p_spread + wind_bonus, 0.0, 1.0)
                            if np.random.rand() < prob:
                                new_grid[r, c] = BURNING
                                break
    return new_grid


def simulate_fire(grid, steps, p_spread=0.6, p_die=0.9, wind=(0, 0)):
    """Simula el incendio durante `steps` pasos. Devuelve todos los estados."""
    states = [grid.copy()]
    for _ in range(steps):
        next_state = forest_step(states[-1], p_spread, p_die, wind)
        states.append(next_state)
        # Detener si no queda fuego activo
        if not np.any(next_state == BURNING):
            break
    return states


def make_forest(n, tree_density=0.7, firebreak_mask=None):
    """
    Genera una grilla forestal aleatoria.

    - tree_density : fracción de celdas con árboles
    - firebreak_mask : grilla binaria NxN; celdas=1 se convierten en EMPTY (cortafuego)
    """
    grid = (np.random.rand(n, n) < tree_density).astype(np.int8)
    if firebreak_mask is not None:
        grid[firebreak_mask == 1] = EMPTY
    return grid


def ignite(grid, position='left'):
    """
    Enciende el fuego en un borde del bosque.
    position: 'left' | 'top' | 'corner'
    """
    g = grid.copy()
    if position == 'left':
        # Primera columna con árboles se enciende
        g[:, 0][g[:, 0] == TREE] = BURNING
    elif position == 'top':
        g[0, :][g[0, :] == TREE] = BURNING
    elif position == 'corner':
        g[0, 0] = BURNING
    return g

In [3]:
# ─── Función de fitness ───────────────────────────────────────────────────────

def fitness_firebreak(firebreak_mask, grid_n, tree_density, p_spread, wind,
                      sim_steps, ignition, n_trials=3, lam=0.5):
    """
    Evalúa qué tan bien protege el bosque un conjunto de cortafuegos.

    El fitness es el promedio de árboles SALVADOS en `n_trials` simulaciones
    estocásticas (promediamos para reducir el ruido del azar).

    Penalización: cada celda de cortafuego tiene un costo (talar árboles
    tiene un coste real). El objetivo real es maximizar árboles salvados
    por unidad de cortafuego utilizada.
    """
    cost = firebreak_mask.sum()
    saved_list = []
    for _ in range(n_trials):
        np.random.seed()
        forest = make_forest(grid_n, tree_density, firebreak_mask)
        forest = ignite(forest, ignition)
        states = simulate_fire(forest, sim_steps, p_spread=p_spread, wind=wind)
        final  = states[-1]

        # Contar solo árboles vivos que NO fueron talados para el cortafuego
        saved = np.sum((final == TREE) & (firebreak_mask == 0))
        saved_list.append(saved)

    avg_saved = np.mean(saved_list)
    penalty   = lam * cost
    return float(avg_saved - penalty)

In [4]:
# ─── Algoritmo Genético ───────────────────────────────────────────────────────
# El cromosoma es una grilla binaria NxN:
#   0 = no hay cortafuego en esta celda
#   1 = cortafuego (se tala el árbol)
# El AG busca la distribución óptima de cortafuegos.

def init_population(pop_size, grid_n, density=0.05):
    """Población inicial con pocos cortafuegos (densidad baja)."""
    return [
        (np.random.rand(grid_n, grid_n) < density).astype(np.uint8)
        for _ in range(pop_size)
    ]


def tournament_selection(population, fitnesses, k=3):
    idx    = np.random.choice(len(population), k, replace=False)
    winner = idx[np.argmax([fitnesses[i] for i in idx])]
    return population[winner].copy()


def uniform_crossover(p1, p2):
    mask = np.random.rand(*p1.shape) < 0.5
    return np.where(mask, p1, p2).astype(np.uint8)


def mutate(individual, rate):
    flip = np.random.rand(*individual.shape) < rate
    return np.bitwise_xor(individual, flip.astype(np.uint8))


def genetic_algorithm_fire(
    grid_n        = 30,
    tree_density  = 0.7,
    p_spread      = 0.6,
    wind          = (0, 1),   # viento hacia la derecha
    sim_steps     = 60,
    ignition      = 'left',
    pop_size      = 40,
    generations   = 40,
    mutation_rate = 0.02,
    elitism       = 2,
    seed          = 42,
    lam           = 0.5,
    callback      = None,
):
    if seed is not None:
        np.random.seed(seed)

    population = init_population(pop_size, grid_n)
    history = {'best': [], 'avg': [], 'best_individuals': []}

    for gen in range(generations):
        fitnesses = [
            fitness_firebreak(ind, grid_n, tree_density, p_spread,
                              wind, sim_steps, ignition, lam=lam)
            for ind in population
        ]

        best_idx = int(np.argmax(fitnesses))
        history['best'].append(fitnesses[best_idx])
        history['avg'].append(float(np.mean(fitnesses)))
        history['best_individuals'].append(population[best_idx].copy())

        if callback:
            callback(gen, fitnesses[best_idx], float(np.mean(fitnesses)))

        sorted_idx = np.argsort(fitnesses)[::-1]
        new_pop    = [population[i].copy() for i in sorted_idx[:elitism]]

        while len(new_pop) < pop_size:
            p1    = tournament_selection(population, fitnesses)
            p2    = tournament_selection(population, fitnesses)
            child = uniform_crossover(p1, p2)
            child = mutate(child, mutation_rate)
            new_pop.append(child)

        population = new_pop

    return history

In [5]:
# ─── Visualización ────────────────────────────────────────────────────────────

def plot_comparison(best_mask, grid_n, tree_density, p_spread, wind,
                    sim_steps, ignition, history, seed=0):
    """
    Muestra lado a lado:
      - Bosque SIN cortafuegos optimizados
      - Bosque CON cortafuegos del mejor individuo
      - Curva de convergencia del AG
    """
    np.random.seed(seed)

    # Simulación sin cortafuegos
    forest_base = make_forest(grid_n, tree_density)
    forest_base = ignite(forest_base, ignition)
    states_base = simulate_fire(forest_base, sim_steps, p_spread=p_spread, wind=wind)

    # Simulación con cortafuegos óptimos
    np.random.seed(seed)
    forest_opt  = make_forest(grid_n, tree_density, best_mask)
    forest_opt  = ignite(forest_opt, ignition)
    states_opt  = simulate_fire(forest_opt, sim_steps, p_spread=p_spread, wind=wind)

    burned_base = np.sum(states_base[-1] == BURNED)
    burned_opt  = np.sum(states_opt[-1]  == BURNED)
    saved       = burned_base - burned_opt
    cost        = best_mask.sum()

    fig = plt.figure(figsize=(15, 10))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

    legend_patches = [
        mpatches.Patch(color='#c2b280', label='Tierra/Cortafuego'),
        mpatches.Patch(color='#2d6a1f', label='Árbol sano'),
        mpatches.Patch(color='#ff4500', label='En llamas'),
        mpatches.Patch(color='#1a1a1a', label='Quemado'),
    ]

    # Estado inicial sin cortafuegos
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.imshow(states_base[0], cmap=CMAP, norm=NORM, interpolation='nearest')
    ax0.set_title('Sin cortafuegos\n(estado inicial)', fontsize=10)
    ax0.axis('off')

    # Estado final sin cortafuegos
    ax1 = fig.add_subplot(gs[0, 1])
    ax1.imshow(states_base[-1], cmap=CMAP, norm=NORM, interpolation='nearest')
    ax1.set_title(f'Sin cortafuegos\n(final — quemados: {burned_base})', fontsize=10)
    ax1.axis('off')

    # Cortafuegos óptimos
    ax2 = fig.add_subplot(gs[0, 2])
    overlay = np.zeros((*best_mask.shape, 4))
    overlay[best_mask == 1] = [1.0, 0.9, 0.0, 0.8]  # amarillo = cortafuego
    ax2.imshow(states_opt[0], cmap=CMAP, norm=NORM, interpolation='nearest')
    ax2.imshow(overlay, interpolation='nearest')
    ax2.set_title(f'Cortafuegos óptimos\n({cost} celdas taladas)', fontsize=10)
    ax2.axis('off')

    # Estado final con cortafuegos
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.imshow(states_opt[-1], cmap=CMAP, norm=NORM, interpolation='nearest')
    ax3.set_title(f'Con cortafuegos\n(final — quemados: {burned_opt})', fontsize=10)
    ax3.axis('off')
    ax3.legend(handles=legend_patches, loc='lower right', fontsize=7)

    # Heatmap de cortafuegos
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.imshow(best_mask, cmap='YlOrBr', interpolation='nearest', vmin=0, vmax=1)
    ax4.set_title('Distribución de cortafuegos\n(cafe = talado)', fontsize=10)
    ax4.axis('off')

    # Convergencia
    ax5 = fig.add_subplot(gs[1, 2])
    gens = range(len(history['best']))
    ax5.plot(gens, history['best'], color='#e63946', lw=2,   label='Mejor')
    ax5.plot(gens, history['avg'],  color='#f4a261', lw=1.5, label='Promedio', ls='--')
    ax5.fill_between(gens, history['avg'], history['best'], alpha=0.15, color='#e63946')
    ax5.set_xlabel('Generación')
    ax5.set_ylabel('Fitness (árboles salvados − costo)')
    ax5.set_title('Convergencia del AG', fontsize=10)
    ax5.legend(fontsize=9)
    ax5.grid(True, lw=0.4, alpha=0.5)
    ax5.spines[['top', 'right']].set_visible(False)

    fig.suptitle(
        f'Optimización de cortafuegos con AG  —  '
        f'Árboles salvados: {saved - cost}  |  Costo: {cost} celdas',
        fontsize=12, fontweight='bold'
    )
    plt.show()


def animate_fire(mask, grid_n, tree_density, p_spread, wind, sim_steps, ignition, seed=0):
    """Anima la propagación del incendio con y sin cortafuegos lado a lado."""
    np.random.seed(seed)
    forest_base = make_forest(grid_n, tree_density)
    forest_base = ignite(forest_base, ignition)
    states_base = simulate_fire(forest_base, sim_steps, p_spread=p_spread, wind=wind)

    np.random.seed(seed)
    forest_opt  = make_forest(grid_n, tree_density, mask)
    forest_opt  = ignite(forest_opt, ignition)
    states_opt  = simulate_fire(forest_opt, sim_steps, p_spread=p_spread, wind=wind)

    max_steps = max(len(states_base), len(states_opt))
    # Extender el último estado si una simulación termina antes
    while len(states_base) < max_steps:
        states_base.append(states_base[-1])
    while len(states_opt) < max_steps:
        states_opt.append(states_opt[-1])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    im1 = ax1.imshow(states_base[0], cmap=CMAP, norm=NORM, interpolation='nearest')
    im2 = ax2.imshow(states_opt[0],  cmap=CMAP, norm=NORM, interpolation='nearest')
    t1  = ax1.set_title('Sin cortafuegos  t=0')
    t2  = ax2.set_title('Con cortafuegos  t=0')
    ax1.axis('off')
    ax2.axis('off')

    def update(frame):
        im1.set_data(states_base[frame])
        im2.set_data(states_opt[frame])
        b1 = np.sum(states_base[frame] == BURNED)
        b2 = np.sum(states_opt[frame]  == BURNED)
        t1.set_text(f'Sin cortafuegos  t={frame}  quemados={b1}')
        t2.set_text(f'Con cortafuegos  t={frame}  quemados={b2}')
        return im1, im2, t1, t2

    ani = animation.FuncAnimation(
        fig, update, frames=max_steps, interval=120, blit=True, repeat=False
    )
    plt.tight_layout()
    plt.close(fig)
    return ani

In [6]:
# ─── Dashboard interactivo ────────────────────────────────────────────────────

w_grid    = widgets.IntSlider(value=30, min=15, max=50, step=5,
                              description='Grilla N×N:',
                              style={'description_width': '110px'},
                              layout=widgets.Layout(width='340px'))
w_density = widgets.FloatSlider(value=0.70, min=0.3, max=1.00, step=0.05,
                                description='Densidad bosque:',
                                readout_format='.2f',
                                style={'description_width': '110px'},
                                layout=widgets.Layout(width='340px'))
w_spread  = widgets.FloatSlider(value=0.60, min=0.1, max=1.00, step=0.05,
                                description='Prob. propagación:',
                                readout_format='.2f',
                                style={'description_width': '110px'},
                                layout=widgets.Layout(width='340px'))
w_wind    = widgets.Dropdown(
                options=[('Sin viento', (0,0)), ('→ Derecha', (0,1)),
                         ('↓ Abajo', (1,0)), ('↗ Diagonal', (1,1))],
                value=(0, 1),
                description='Viento:',
                style={'description_width': '110px'},
                layout=widgets.Layout(width='280px'))
w_ignite  = widgets.Dropdown(
                options=[('Borde izquierdo', 'left'),
                         ('Borde superior', 'top'),
                         ('Esquina', 'corner')],
                value='left',
                description='Ignición:',
                style={'description_width': '110px'},
                layout=widgets.Layout(width='280px'))
w_pop     = widgets.IntSlider(value=40, min=20, max=100, step=10,
                              description='Población AG:',
                              style={'description_width': '110px'},
                              layout=widgets.Layout(width='340px'))
w_gens    = widgets.IntSlider(value=40, min=10, max=100, step=10,
                              description='Generaciones:',
                              style={'description_width': '110px'},
                              layout=widgets.Layout(width='340px'))
w_mut     = widgets.FloatSlider(value=0.02, min=0.001, max=0.10, step=0.005,
                                description='Mutación:',
                                readout_format='.3f',
                                style={'description_width': '110px'},
                                layout=widgets.Layout(width='340px'))
w_seed    = widgets.IntText(value=42, description='Semilla:',
                            style={'description_width': '110px'},
                            layout=widgets.Layout(width='200px'))
w_lambda = widgets.FloatSlider(value=0.5, min=0.0, max=3.0, step=0.1,
                               description='λ costo:',
                               style={'description_width': '110px'},
                               layout=widgets.Layout(width='340px'))
w_run     = widgets.Button(description='▶  Ejecutar AG',
                           button_style='danger',
                           layout=widgets.Layout(width='160px', height='36px'))
w_anim    = widgets.Button(description='🎬  Animar incendio',
                           button_style='info',
                           layout=widgets.Layout(width='170px', height='36px'),
                           disabled=True)
w_prog    = widgets.IntProgress(value=0, min=0, max=100,
                                bar_style='danger',
                                layout=widgets.Layout(width='100%'))
w_status  = widgets.Label(value='Configura los parámetros y presiona Ejecutar.')
w_out     = widgets.Output()

_state = {'history': None, 'params': {}}


def on_run(_):
    w_run.disabled  = True
    w_anim.disabled = True
    w_prog.value    = 0
    total_gens      = w_gens.value

    def cb(gen, best, avg):
        w_prog.value   = int((gen + 1) / total_gens * 100)
        w_status.value = (
            f'Generación {gen+1}/{total_gens}  ·  '
            f'Mejor fitness: {best:.1f}  ·  Promedio: {avg:.1f}'
        )

    with w_out:
        clear_output(wait=True)
        print('Ejecutando AG...')

    params = dict(
        grid_n       = w_grid.value,
        tree_density = w_density.value,
        p_spread     = w_spread.value,
        wind         = w_wind.value,
        sim_steps    = 60,
        ignition     = w_ignite.value,
        pop_size     = w_pop.value,
        generations  = total_gens,
        mutation_rate= w_mut.value,
        seed         = w_seed.value,
        lam          = w_lambda.value,
        callback     = cb,
    )

    history = genetic_algorithm_fire(**params)
    _state['history'] = history
    _state['params']  = params

    best_mask = history['best_individuals'][-1]

    with w_out:
        clear_output(wait=True)
        plot_comparison(
            best_mask,
            grid_n       = params['grid_n'],
            tree_density = params['tree_density'],
            p_spread     = params['p_spread'],
            wind         = params['wind'],
            sim_steps    = params['sim_steps'],
            ignition     = params['ignition'],
            history      = history,
            seed         = params['seed'],
        )

    w_status.value  = f'✓ Listo. Mejor fitness: {history["best"][-1]:.1f}'
    w_run.disabled  = False
    w_anim.disabled = False


def on_animate(_):
    if _state['history'] is None:
        return
    p = _state['params']
    best_mask = _state['history']['best_individuals'][-1]
    with w_out:
        clear_output(wait=True)
        w_status.value = 'Generando animación...'
        ani = animate_fire(
            best_mask,
            grid_n       = p['grid_n'],
            tree_density = p['tree_density'],
            p_spread     = p['p_spread'],
            wind         = p['wind'],
            sim_steps    = p['sim_steps'],
            ignition     = p['ignition'],
            seed         = p['seed'],
        )
        _state['_ani'] = ani
        display(HTML(ani.to_jshtml()))
        w_status.value = '✓ Animación lista.'


w_run.on_click(on_run)
w_anim.on_click(on_animate)

col1 = widgets.VBox([w_grid, w_density, w_spread, w_wind, w_ignite])
col2 = widgets.VBox([w_pop, w_gens, w_mut, w_lambda, w_seed,
                     widgets.HBox([w_run, w_anim])])

ui = widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 8px">🔥 Optimización de cortafuegos con Algoritmos Genéticos</h3>'),
    widgets.HBox([col1, col2], layout=widgets.Layout(gap='32px')),
    w_prog,
    w_status,
    w_out,
])

display(ui)

fitness = avg_saved - (λ * cost)

Donde negativo significa que estan quemandose mas de los que se estan salvando. Cost esta dado por la cantidad de arboles talados.

El valor de lambda es la que dicta que objetivo buscamos:

- λ = 0.0 — el costo no importa en absoluto. El AG talará todo el bosque si así salva más árboles. Solución óptima trivial: cortafuego total.
- λ = 0.5 — talar una celda "vale" la mitad que salvar una. El AG solo tala si por cada celda talada salva al menos 2 árboles.
- λ = 1.0 — talar una celda cuesta lo mismo que salvar una. Solo vale la pena si hay ganancia neta directa.
- λ = 2.0 — talar es muy caro. El AG buscará soluciones minimalistas con muy pocos cortafuegos muy bien ubicados.